# Marconi CoastCam image scan (step 1 of the waterline DEM workflow)
csherwood@usgs.gov, 2026-10-21, generated with Claude Opus 5.5

For every timex image during the nanopod record: calibration files from the lookup table, image quality index
(0 = night, 1 = too blurry, 2 = sharp but sunglint, 4 = sharp, high quality), waves and water level interpolated
to the image time, deep-water wave height by reverse shoaling (linear theory), Stockdon et al. (2006) setup and R2
with a starting slope of 0.12, and two camera-pointing checks:
* **horizon**: observed sea horizon vs. the horizon predicted from the calibration EO (offset in px, fitted tilt/roll changes);
* **features**: rotation (azimuth, tilt, roll) of each image relative to a reference image for that camera/EO,
  from SIFT matches of static features (Kabsch rotation inside RANSAC). The reference is the first quality-4 image
  whose horizon matches the calibration within `hz_ref` px.

Changes are listed in `eo_changes.csv` but **not applied**.

Requires: numpy, pandas, scipy, opencv (with SIFT, i.e. OpenCV >= 4.4), pyyaml, matplotlib.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import marconi_scan as ms

# --- config
PROJ = Path(r'F:\crs\proj\2026_wl_dems')
WAVES = PROJ / 'nanopod_waves' / 'sig1000_waves_ALL1.csv'
IMG_ROOT = Path(r'F:\crs\proj\marconi_imagery')
CAL_DIR = IMG_ROOT / 'calibration_parameters'
CAL_TABLE = CAL_DIR / 'calib_parameter_table.csv'
STATIONS = ['caco-03', 'caco-04', 'caco-05']
OUT_DIR = PROJ / 'scan'
FIG_DIR = PROJ / 'fig'
SITE_LAT, SITE_LON = 41.891, -69.963     # Marconi CoastCam, for sun position
BETA0 = 0.12                             # initial foreshore slope for Stockdon
TH = ms.THRESH | {}                      # override thresholds here, e.g. {'blur': 25.}
N_SAMPLE = 120                           # images for the threshold check

OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
SCAN_CSV = OUT_DIR / 'image_scan.csv'

In [ ]:
waves = pd.read_csv(WAVES, parse_dates=['time'])
FIRST, LAST = waves.time.iloc[0], waves.time.iloc[-1]
table = ms.read_calib_table(CAL_TABLE)
imgs = ms.list_timex(IMG_ROOT, STATIONS, t0=FIRST, t1=LAST)
imgs['eo_file'] = [ms.calib_files(table, t, s, c)[0] for t, s, c in zip(imgs.time, imgs.station, imgs.camera)]
print(f'nanopod record {FIRST} to {LAST}: {len(imgs)} timex images')
print(imgs.groupby(['station', 'camera', 'eo_file'], dropna=False).time.agg(['count', 'min', 'max']))

## Threshold check
Quality metrics on a random sample of images (no pointing). Check the scatter and the example images, then
adjust `TH` in the config cell before the full scan.

In [ ]:
smp = imgs.dropna(subset=['eo_file']).sample(min(N_SAMPLE, imgs.eo_file.notna().sum()), random_state=1).sort_values('time')
smp['sun_el'], smp['sun_az'] = ms.sun_position(smp.time, SITE_LAT, SITE_LON)
rows = []
for _, r in smp.iterrows():
    eo_file, io_file = ms.calib_files(table, r.time, r.station, r.camera)
    io, eo = ms.read_yaml(CAL_DIR / io_file), ms.read_yaml(CAL_DIR / eo_file)
    rows.append(ms.scan_image(r.path, io, eo, r.sun_el, r.sun_az, TH)[0] | {'path': r.path, 'camera': r.camera})
smp_m = pd.DataFrame(rows)
print(smp_m.groupby('quality').size())

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for q, c in zip([0, 1, 2, 4], ['k', 'tab:gray', 'tab:orange', 'tab:blue']):
    s = smp_m[smp_m.quality == q]
    ax[0].scatter(s.brightness, s.sharpness, s=12, c=c, label=str(q))
    ax[1].scatter(s.sharpness, s.glint_frac, s=12, c=c, label=str(q))
ax[0].axvline(TH['dark'], c='k', lw=0.8); ax[0].axhline(TH['blur'], c='k', lw=0.8)
ax[1].axvline(TH['blur'], c='k', lw=0.8); ax[1].axhline(TH['glint'], c='k', lw=0.8)
ax[0].set_xlabel('Mean brightness (0-255)'); ax[0].set_ylabel('Sharpness (Laplacian variance)'); ax[0].set_yscale('log')
ax[1].set_xlabel('Sharpness (Laplacian variance)'); ax[1].set_ylabel('Saturated fraction in sea band'); ax[1].set_xscale('log')
ax[0].legend(title='quality', fontsize=8)
for a, l in zip(ax, 'ab'):
    a.text(0.02, 0.97, l, transform=a.transAxes, va='top')
fig.tight_layout()
fig.savefig(FIG_DIR / 'scan_thresholds.png', dpi=150)

In [ ]:
# up to 4 examples of each quality class
ex = smp_m.groupby('quality').head(4).sort_values('quality').reset_index(drop=True)
nc = 4
nr = int(np.ceil(len(ex) / nc))
fig, ax = plt.subplots(nr, nc, figsize=(3.2 * nc, 2.8 * nr), squeeze=False)
for i, a in enumerate(ax.flat):
    a.axis('off')
    if i < len(ex):
        im = cv2.cvtColor(cv2.resize(cv2.imread(ex.path[i]), None, fx=0.15, fy=0.15), cv2.COLOR_BGR2RGB)
        a.imshow(im)
        a.set_title(f'q {ex.quality[i]:.0f}, sharp {ex.sharpness[i]:.0f}, glint {ex.glint_frac[i]:.3f}', fontsize=8)
        a.text(0.02, 0.97, chr(97 + i), transform=a.transAxes, va='top', color='w')
fig.tight_layout()
fig.savefig(FIG_DIR / 'scan_quality_examples.jpg', dpi=120)

## Full scan
Writes `image_scan.csv` in chunks and resumes where it stopped if rerun. About 1 s per daytime image when the
pointing check runs (every `TH['pt_every']` per camera), much less otherwise.

In [ ]:
scan = ms.scan(imgs.drop(columns='eo_file'), waves, table, CAL_DIR, SCAN_CSV, SITE_LAT, SITE_LON, beta=BETA0, th=TH)
print(scan.groupby(['camera', 'quality']).size().unstack())
print('reference images:'); print(scan.loc[scan.is_ref == True, ['camera', 'eo_file', 'time', 'hz_offset', 'path']])

## Running list of camera-pointing changes (not applied)
Daily medians of the feature-based changes, grouped into runs of constant pointing. `azimuth`, `tilt`, `roll` are
the calibration angles plus the run median change; `hz_dtilt`, `hz_droll` are the independent horizon estimates
(tilt is reliable from the horizon; roll is partly confounded by azimuth for these oblique, rolled views).

In [ ]:
chg = ms.eo_changes(scan, CAL_DIR, tol=0.15, min_inliers=TH['pt_min_inliers'])
chg.to_csv(OUT_DIR / 'eo_changes.csv', index=False, float_format='%.4f')
chg

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
ax[0].plot(waves.time, waves.wh_4061, c='tab:blue', lw=0.8, label='Hs at nanopod')
ax[0].plot(scan.time, scan.R2, '.', c='tab:red', ms=2, label=f'R2 (Stockdon, slope {BETA0})')
ax[0].set_ylabel('Height (m)'); ax[0].legend(fontsize=8)
ax[1].plot(waves.time, waves.water_level, c='k', lw=0.8)
ax[1].set_ylabel('Water level (m)')
for cam, off in [('c1', -0.1), ('c2', 0.1)]:
    s = scan[scan.camera == cam]
    ax[2].plot(s.time, s.quality + off, '.', ms=2, label=cam)
    ax[3].plot(s.time, s.d_az, '.', ms=2, label=f'{cam} azimuth')
    ax[3].plot(s.time, s.d_tilt, '.', ms=2, label=f'{cam} tilt')
    ax[3].plot(s.time, s.d_roll, '.', ms=2, label=f'{cam} roll')
ax[2].set_yticks([0, 1, 2, 4]); ax[2].set_ylabel('Quality index'); ax[2].legend(fontsize=8, markerscale=4)
ax[3].set_ylabel('Pointing change (deg)'); ax[3].legend(fontsize=7, ncol=3, markerscale=4)
for a, l in zip(ax, 'abcd'):
    a.text(0.01, 0.95, l, transform=a.transAxes, va='top')
fig.tight_layout()
fig.savefig(FIG_DIR / 'scan_timeseries.png', dpi=150)